# 🎨 Fooocus Studio - Google Colab

Entorno optimizado de generación y renderizado de alta fidelidad con **Fooocus**:
- 🧠 **Sincronización Inteligente con Drive:** Busca primero en tu Google Drive (`MyDrive/RuinedFooocus`). Si el modelo elegido no está en tu Drive, lo descarga y **lo guarda automáticamente en tu Drive** para no volver a descargarlo nunca más.
- 🎯 **Descarga de 1 Solo Modelo:** Selecciona en el menú desplegable cuál modelo deseas utilizar.
- ⚡ **Copia Anti-RAM:** Transfiere tus modelos a la GPU sin saturar la memoria RAM.
- 💾 **Almacenamiento Directo:** Todas las imágenes generadas se guardan automáticamente en tu Google Drive.

In [ ]:
# @title 🚀 Selección de Modelo y Lanzamiento de Fooocus
# @markdown ### 🎯 Elige el modelo que deseas usar (se buscará en tu Drive y si no está, se descargará y guardará allí):
Modelo_Seleccionado = "CyberRealistic PONY v6.5 (Top Hiper-Realismo & Anatomia Sin Censura)" #@param ["CyberRealistic PONY v6.5 (Top Hiper-Realismo & Anatomia Sin Censura)", "CyberRealistic XL Play v5.0 (Fotorealismo General)", "Pony Diffusion V6 XL (Arte, Poses & Anatomia)", "Juggernaut XL v9 Photo (Estudio & Texturas)", "RealVisXL v5.0 (Fotografia Raw)", "Ninguno (Usar solo los modelos de mi Drive)"]

# @markdown ### ⚡ Extensiones de Detalle & Iluminación (LoRAs):
Descargar_Loras_Realismo = True #@param {type:"boolean"}

import os
import sys
from tqdm import tqdm

# 1. Repositorio
%cd /content
if not os.path.exists('/content/Fooocus'):
    print("\n📦 Preparando entorno de Fooocus...")
    !git clone https://github.com/Christianebg1/Fooocus.git
    %cd /content/Fooocus
else:
    %cd /content/Fooocus
    !git pull

local_checkpoints = '/content/Fooocus/models/checkpoints'
local_loras = '/content/Fooocus/models/loras'
os.makedirs(local_checkpoints, exist_ok=True)
os.makedirs(local_loras, exist_ok=True)

# 2. Conectar Google Drive y crear carpetas
drive_base = '/content/drive/MyDrive/RuinedFooocus'
drive_checkpoints = os.path.join(drive_base, 'checkpoints')
drive_loras = os.path.join(drive_base, 'loras')
drive_outputs = os.path.join(drive_base, 'outputs')

drive_connected = False
try:
    from google.colab import drive
    print("\n1️⃣ Conectando Google Drive...")
    drive.mount('/content/drive')
    if os.path.exists('/content/drive/MyDrive'):
        drive_connected = True
        os.makedirs(drive_checkpoints, exist_ok=True)
        os.makedirs(drive_loras, exist_ok=True)
        os.makedirs(drive_outputs, exist_ok=True)
        !rm -rf /content/Fooocus/outputs
        !ln -s "{drive_outputs}" /content/Fooocus/outputs
        print("✅ Google Drive vinculado. Las imágenes se guardarán en MyDrive/RuinedFooocus/outputs")
except Exception as e:
    print("ℹ️ Google Drive no conectado. Las imágenes se guardarán temporalmente en Colab.")

# 3. Función de copia Anti-RAM
def copiar_archivos_sin_ram(origen, destino):
    if not os.path.exists(origen):
        return 0
    copiados = 0
    for archivo in os.listdir(origen):
        ruta_origen = os.path.join(origen, archivo)
        ruta_destino = os.path.join(destino, archivo)
        if os.path.isfile(ruta_origen) and not os.path.exists(ruta_destino):
            peso_total = os.path.getsize(ruta_origen)
            print(f"\n📥 Copiando desde Drive: {archivo}")
            with open(ruta_origen, 'rb') as fsrc, open(ruta_destino, 'wb') as fdst, tqdm(
                total=peso_total, unit='B', unit_scale=True, unit_divisor=1024,
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
            ) as pbar:
                while True:
                    buf = fsrc.read(1024 * 1024 * 16)
                    if not buf:
                        break
                    fdst.write(buf)
                    os.posix_fadvise(fsrc.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
                    os.posix_fadvise(fdst.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
                    pbar.update(len(buf))
            copiados += 1
    return copiados

# Helper para descargar y guardar automáticamente en Drive
def descargar_y_respaldar_en_drive(url, archivo_nombre, destino_local, destino_drive):
    ruta_local = os.path.join(destino_local, archivo_nombre)
    if not os.path.exists(ruta_local):
        print(f"\n🌐 Descargando {archivo_nombre}...")
        !wget -c "{url}" -O "{ruta_local}"
        if drive_connected and os.path.exists(destino_drive):
            ruta_drive = os.path.join(destino_drive, archivo_nombre)
            if not os.path.exists(ruta_drive):
                print(f"💾 Guardando copia en tu Google Drive ({archivo_nombre})...")
                with open(ruta_local, 'rb') as fsrc, open(ruta_drive, 'wb') as fdst, tqdm(
                    total=os.path.getsize(ruta_local), unit='B', unit_scale=True, unit_divisor=1024,
                    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]'
                ) as pbar:
                    while True:
                        buf = fsrc.read(1024 * 1024 * 16)
                        if not buf:
                            break
                        fdst.write(buf)
                        os.posix_fadvise(fsrc.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
                        os.posix_fadvise(fdst.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
                        pbar.update(len(buf))
                print(f"✅ Modelo guardado permanentemente en tu Drive.")

print("\n2️⃣ Buscando modelos y extensiones en tu Google Drive...")
if drive_connected and os.path.exists(drive_checkpoints):
    copiar_archivos_sin_ram(drive_checkpoints, local_checkpoints)
    copiar_archivos_sin_ram(drive_loras, local_loras)

# 4. Procesar el modelo seleccionado por el usuario
catalogo_modelos = {
    "CyberRealistic PONY v6.5 (Top Hiper-Realismo & Anatomia Sin Censura)": (
        "CyberRealisticPony_v6.5_FP16.safetensors",
        "https://huggingface.co/cyberdelia/CyberRealisticPony/resolve/main/CyberRealisticPony_v6.5_FP16.safetensors"
    ),
    "CyberRealistic XL Play v5.0 (Fotorealismo General)": (
        "CyberRealisticXLPlay_V5_FP16.safetensors",
        "https://huggingface.co/cyberdelia/CyberRealisticXL/resolve/main/CyberRealisticXLPlay_V5_FP16.safetensors"
    ),
    "Pony Diffusion V6 XL (Arte, Poses & Anatomia)": (
        "ponyDiffusionV6XL_v6.safetensors",
        "https://huggingface.co/AstraliteHeart/pony-diffusion-v6-xl/resolve/main/v6.safetensors"
    ),
    "Juggernaut XL v9 Photo (Estudio & Texturas)": (
        "Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors",
        "https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors"
    ),
    "RealVisXL v5.0 (Fotografia Raw)": (
        "RealVisXL_V5.0.safetensors",
        "https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0.safetensors"
    )
}

if Modelo_Seleccionado in catalogo_modelos:
    nombre_archivo, url_descarga = catalogo_modelos[Modelo_Seleccionado]
    if os.path.exists(os.path.join(local_checkpoints, nombre_archivo)):
        print(f"\n✅ El modelo '{nombre_archivo}' ya está listo (cargado desde Drive o local).")
    else:
        descargar_y_respaldar_en_drive(url_descarga, nombre_archivo, local_checkpoints, drive_checkpoints)

# Descargar LoRA de detalle si está activo
if Descargar_Loras_Realismo:
    if not os.path.exists(os.path.join(local_loras, 'add-detail-xl.safetensors')):
        descargar_y_respaldar_en_drive(
            'https://huggingface.co/nerfgun3/add-detail-xl/resolve/main/add-detail-xl.safetensors',
            'add-detail-xl.safetensors',
            local_loras,
            drive_loras
        )

# Asegurar que al menos un modelo esté disponible
checkpoints_activos = [f for f in os.listdir(local_checkpoints) if f.endswith(('.safetensors', '.ckpt'))]
if not checkpoints_activos:
    print("\n🌐 No se detectó ningún modelo en Drive. Descargando CyberRealistic PONY v6.5 por defecto...")
    descargar_y_respaldar_en_drive(
        'https://huggingface.co/cyberdelia/CyberRealisticPony/resolve/main/CyberRealisticPony_v6.5_FP16.safetensors',
        'CyberRealisticPony_v6.5_FP16.safetensors',
        local_checkpoints,
        drive_checkpoints
    )

# 5. Dependencias y Limpieza
print("\n3️⃣ Verificando dependencias...")
!pip uninstall -y cupy-cuda12x cupy cupy-cuda11x 2>/dev/null
!pip install --prefer-binary --only-binary=:all: torchsde pytorch_lightning gradio==3.41.2 opencv-contrib-python-headless onnxruntime rembg segment_anything gradio-client==0.5.0 aiofiles ffmpy supervision

# 6. Lanzar Fooocus
print("\n4️⃣ Lanzando Fooocus...")
os.environ["LAUNCH_LIVE_OUTPUT"] = "1"
!python launch.py --share --always-high-vram --disable-preset-download
